# Task 2 - Bayesian Change Point Modeling of Brent Oil Prices

This notebook builds a Bayesian change point model in **PyMC** to detect and
quantify structural breaks in Brent crude oil prices, then associates the
detected change points with major geopolitical / economic / OPEC events.

**Approach**
1. Load the cleaned daily price series and compute log returns.
2. Aggregate to monthly averages for tractable, interpretable change point detection.
3. Fit a single change point model: `tau ~ DiscreteUniform`, means `mu_1`, `mu_2`
   selected with `pm.math.switch`, `Normal` likelihood, sampled with `pm.sample()`.
4. Check convergence (`r_hat`), plot the posterior of `tau`, and quantify the impact.
5. Detect multiple change points (ruptures) and associate all breaks with events.
6. Compute per-event before/after price and volatility impacts.
7. Export results for the dashboard.

In [1]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import json
import arviz as az
import pandas as pd

from src.data_loader import add_log_returns, load_events, load_prices
from src import change_point_model as cp

pd.set_option("display.max_columns", None)

## 1. Load data and aggregate to monthly

In [2]:
df = add_log_returns(load_prices())
events = load_events()
monthly = cp.to_monthly(df)
print(f"Daily rows: {len(df):,} | Monthly points: {len(monthly)}")
monthly.tail()

2026-07-16 09:53:45,448 | INFO    | src.data_loader | Loaded 9011 price rows (0 dropped) spanning 1987-05-20 to 2022-11-14.


2026-07-16 09:53:45,475 | INFO    | src.data_loader | Loaded 15 curated key events.


Daily rows: 9,011 | Monthly points: 427


,Date,Price
422,2022-07-31,111.925714
423,2022-08-31,100.446364
424,2022-09-30,89.764762
425,2022-10-31,93.331905
426,2022-11-30,95.999000


## 2. Bayesian single change point model (PyMC)

```
tau  ~ DiscreteUniform(0, N-1)
mu_1 ~ Normal(mean, 2*std)     # regime before tau
mu_2 ~ Normal(mean, 2*std)     # regime after tau
sigma ~ HalfNormal(2*std)
mu = switch(tau >= t, mu_1, mu_2)
price ~ Normal(mu, sigma)
```

In [3]:
# Draws/tune can be increased for publication-quality posteriors.
# Results are also produced by: python scripts/run_change_point.py
model, trace = cp.build_and_sample(
    monthly["Price"], monthly["Date"], draws=1000, tune=1000, chains=2
)

2026-07-16 09:53:45,596 | INFO    | pytensor.configparser | Suppressed KeyError in PyTensorConfigParser.add for parameter 'cxx'!


2026-07-16 09:53:45,598 | INFO    | pytensor.configparser | Suppressed KeyError in PyTensorConfigParser.add for parameter 'gcc_version_str'!


2026-07-16 09:53:45,600 | INFO    | pytensor.configparser | Suppressed KeyError in PyTensorConfigParser.add for parameter 'compile__timeout'!


2026-07-16 09:53:45,602 | INFO    | pytensor.configparser | Suppressed KeyError in PyTensorConfigParser.add for parameter 'DebugMode__check_c'!


2026-07-16 09:53:45,609 | INFO    | pytensor.configparser | Suppressed KeyError in PyTensorConfigParser.add for parameter 'compiledir'!


...

2026-07-16 09:53:55,554 | INFO    | pymc.sampling.mcmc | Multiprocess sampling (4 chains in 2 jobs)


2026-07-16 09:53:55,558 | INFO    | pymc.sampling.mcmc | CompoundStep


2026-07-16 09:53:55,567 | INFO    | pymc.sampling.mcmc | >Metropolis: [tau]


2026-07-16 09:53:55,570 | INFO    | pymc.sampling.mcmc | >NUTS: [mu_1, mu_2, sigma]


2026-07-16 09:56:15,200 | INFO    | pymc.sampling.mcmc | Sampling 4 chains for 2_000 tune and 2_000 draw iterations (8_000 + 8_000 draws total) took 140 seconds.


### 2.1 Convergence check

We want `r_hat` values close to 1.0.

In [4]:
summary = az.summary(trace, var_names=["tau", "mu_1", "mu_2", "sigma"])
summary

,mean,sd,eti89_lb,eti89_ub,ess_bulk,ess_tail,r_hat,mcse_mean,mcse_sd
tau,213.2,2,210,220,1425,1351,1.00,0.055,0.055
mu_1,21.44,1.3,19,24,5279,5247,1.00,0.018,0.013
mu_2,75.78,1.33,74,78,6142,5764,1.00,0.017,0.012
sigma,18.66,0.66,18,20,8552,6345,1.00,0.0071,0.005


In [5]:
cp.plot_trace(trace, ["tau", "mu_1", "mu_2", "sigma"])
print("Saved trace plot to reports/figures/cp_trace.png")

Saved trace plot to reports/figures/cp_trace.png


### 2.2 Identify the change point and quantify the impact

In [6]:
result = cp.summarize_result(trace, monthly["Date"])
cp.plot_tau_posterior(trace, monthly["Date"])
print(f"Change point date (median tau): {result.tau_date.date()}")
print(f"95% credible interval: {result.tau_hdi[0].date()} to {result.tau_hdi[1].date()}")
print(f"Mean price before: ${result.mu_before:.2f}  (95% HDI {result.mu_before_hdi[0]:.1f}-{result.mu_before_hdi[1]:.1f})")
print(f"Mean price after:  ${result.mu_after:.2f}  (95% HDI {result.mu_after_hdi[0]:.1f}-{result.mu_after_hdi[1]:.1f})")
print(f"Change: {result.pct_change:+.1f}%   P(mu_2 > mu_1) = {result.prob_increase:.3f}")

Change point date (median tau): 2005-02-28
95% credible interval: 2004-10-31 to 2005-06-30
Mean price before: $21.44  (95% HDI 18.9-24.0)
Mean price after:  $75.78  (95% HDI 73.2-78.4)
Change: +253.4%   P(mu_2 > mu_1) = 1.000


## 3. Multiple change points (ruptures) and regime overlay

The single Bayesian change point captures the dominant structural break. To
associate several events we also detect multiple breaks with `ruptures` (a fast
complementary method) and overlay them on the price series.

In [7]:
rupt_idx = cp.ruptures_change_points(monthly["Price"], n_bkps=5)
rupt_dates = [monthly["Date"].iloc[i] for i in rupt_idx]
print("Ruptures change point dates:", [d.date().isoformat() for d in rupt_dates])
cp.plot_price_with_changepoints(monthly, result, extra_cp_dates=rupt_dates)
print("Saved regime plot to reports/figures/cp_price_regimes.png")

Ruptures change point dates: ['1999-11-30', '2005-04-30', '2011-02-28', '2014-11-30', '2021-07-31']


Saved regime plot to reports/figures/cp_price_regimes.png


## 4. Associate change points with events

In [8]:
nearest = cp.associate_events(result.tau_date, events, window_days=365)
print("Events nearest the Bayesian change point:")
nearest[["event_date", "event_name", "category", "days_from_tau"]].head(5)

Events nearest the Bayesian change point:


,event_date,event_name,category,days_from_tau
4,2003-03-20,US-led invasion of Iraq,Conflict,-711
5,2008-07-11,2008 oil price peak,Economic,1229
3,2001-09-11,September 11 attacks,Conflict,-1266
6,2008-09-15,Global Financial Crisis (Lehman collapse),Economic,1295
7,2011-02-15,Arab Spring & Libyan civil war,Conflict,2178


## 5. Per-event quantitative impact (90-day windows)

For each curated event, compare the average price and volatility in the 90 days
before vs. after the event.

In [9]:
impact = cp.event_impact_windows(df, events, window_days=90)
impact

,event_date,event_name,category,price_before,price_after,price_pct_change,vol_before,vol_after
0,1990-08-02,Iraq invades Kuwait (Gulf War),Conflict,16.37,32.79,100.3,0.0257,0.0539
1,1997-07-02,Asian Financial Crisis,Economic,18.06,18.51,2.5,0.0188,0.0157
2,1998-12-01,OPEC overproduction price collapse,OPEC,12.35,10.40,-15.8,0.0279,0.0341
3,2001-09-11,September 11 attacks,Conflict,25.77,20.78,-19.4,0.0214,0.0460
4,2003-03-20,US-led invasion of Iraq,Conflict,31.95,26.14,-18.2,0.0227,0.0249
5,2008-07-11,2008 oil price peak,Economic,125.95,108.92,-13.5,0.0217,0.0281
6,2008-09-15,Global Financial Crisis (Lehman collapse),Economic,121.14,65.74,-45.7,0.0249,0.0466
7,2011-02-15,Arab Spring & Libyan civil war,Conflict,93.65,116.47,24.4,0.0113,0.0206
8,2014-11-27,OPEC declines to cut output,OPEC,88.70,56.35,-36.5,0.0123,0.0289
9,2016-01-20,2016 oil price trough,Economic,39.55,35.67,-9.8,0.0244,0.0376


## 6. Export results for the dashboard

In [10]:
results = {
    "generated_at": pd.Timestamp.now().isoformat(),
    "data": {
        "n_daily": int(len(df)),
        "n_monthly": int(len(monthly)),
        "start_date": df["Date"].min().date().isoformat(),
        "end_date": df["Date"].max().date().isoformat(),
    },
    "bayesian_change_point": {
        "tau_date": result.tau_date.date().isoformat(),
        "tau_hdi": [result.tau_hdi[0].date().isoformat(), result.tau_hdi[1].date().isoformat()],
        "mu_before": round(result.mu_before, 2),
        "mu_after": round(result.mu_after, 2),
        "mu_before_hdi": [round(x, 2) for x in result.mu_before_hdi],
        "mu_after_hdi": [round(x, 2) for x in result.mu_after_hdi],
        "pct_change": round(result.pct_change, 1),
        "prob_increase": round(result.prob_increase, 4),
    },
    "ruptures_change_points": [d.date().isoformat() for d in rupt_dates],
    "event_impact": impact.to_dict(orient="records"),
}
out_path = PROJECT_ROOT / "reports" / "model_results.json"
out_path.write_text(json.dumps(results, indent=2))
print(f"Wrote {out_path}")

Wrote C:\Users\USER\Documents\Projects\10 academy\brent-oil-change-point-analysis\reports\model_results.json


## 7. Interpretation

Results from `python scripts/run_change_point.py` (also reproduced by this notebook):

- **Dominant structural break (2005-02-28):** The Bayesian model places τ in
  Feb 2005 (95% CI: Oct 2004 – Jun 2005) with excellent convergence (`r_hat ≈ 1.0`)
  and `P(μ₂ > μ₁) = 1.0`. Mean monthly price shifts from **$21.48/bbl** to
  **$75.78/bbl** — a **+252.8%** regime change. This aligns with the mid-2000s
  demand super-cycle (emerging-market growth, tight spare capacity).
- **Multiple breaks (ruptures):** Additional breaks near Nov 1999, Apr 2005,
  Feb 2011 (Arab Spring), Nov 2014 (OPEC non-cut), and Jul 2021.
- **Quantified event impacts (90-day windows):** Gulf War **+100.3%**;
  Lehman **−45.7%**; OPEC 2014 non-cut **−36.5%**; COVID crash **−57.9%**;
  Russia–Ukraine **+31.1%**.
- **Caveat:** Temporal alignment is association, not proven causation
  (see `reports/assumptions_and_limitations.md`).